In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (root_mean_squared_error, mean_absolute_error, r2_score)

In [4]:
df = pd.read_csv("D:\\mlops_day1\\data\\data.csv")
df.head()

,TV,radio,newspaper,sales
1,230.1,37.8,69.2,22.1
2,44.5,39.3,45.1,10.4
3,17.2,45.9,69.3,9.3
4,151.5,41.3,58.5,18.5
5,180.8,10.8,58.4,12.9


In [5]:
X = df[["TV", "radio", "newspaper"]]
y = df["sales"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=56) 

First, we configure MLflow to store all experiment tracking metadata inside a local SQLite database named mlflow.db located in the current working directory

In [6]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

Then we create an experiment named **Advertising Sales Prediction**

In [7]:
mlflow.set_experiment("Advertising Sales Prediction")

<Experiment: artifact_location='file:d:/mlops_day1/notebooks/mlruns/1', creation_time=1788450633443, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788450633443, lifecycle_stage='active', name='Advertising Sales Prediction', tags={}, trace_location=None, workspace='default'>

Now we create a **run** inside the experiment **Advertising Sales Precition**

In [8]:
with mlflow.start_run(run_name="LinearRegression"):
    model = LinearRegression()

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    #Parameters
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    #Metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

**Lets's open the mlflow user interface**

Open a terminal in the environment where MLflow is installed and run:
                               mlflow ui

Now we create another run using Ridge regression model

In [9]:
with mlflow.start_run(run_name="Ridge Regression"):

    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse= root_mean_squared_error(y_test, y_pred)
    r2= r2_score(y_test, y_pred)

    #Log the parameters
    mlflow.log_param("model_type", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 56)

    #Log the metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    #Log Artifacts
    mlflow.sklearn.log_model(sk_model = model, name="Ridge_Reg_Model")

Artifacts are the concrete output files generated by a run

Now, we use the autologging feature in MLflow

Autologging allows MLflow to automatically capture much of the inforamtion

In [10]:
#Turn on scikit learn autologging
mlflow.sklearn.autolog()

In [11]:
with mlflow.start_run(run_name="Random Forest Autolog") as run:

    model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)

    test_pred = model.predict(X_test)

    test_rmse = root_mean_squared_error(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_r2 = r2_score(y_test, test_pred)

    #Custom project metrics
    mlflow.log_metrics({
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

2026/09/06 08:10:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


### Registering the Model
The MLflow model registry is a central hub file to keep all production ready models in one shared searchable place instead of scattered folders or runs

Among all of the experiments you perform, register the final selected model

In a prediction environment, the models are **continuously trained**. This means the registered models would have many version:

- **Advertising_Sales_Model**
  - Version 1
  - Version 2
  - Version 3
  - Version 4

In [12]:
# for registering the model, we require the model URI
# URI is a unique idnetifier for model

run_id = run.info.run_id
model_uri = f"runs:/{run_id}/model"
print(model_uri)

runs:/9edcfb2bcfa54450a821275b8865c8e0/model


Now let's register the model

In [13]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="Advertising_Sales_Model"
)
registered_model

Registered model 'Advertising_Sales_Model' already exists. Creating a new version of this model...
2026/09/06 08:11:08 WARNING mlflow.tracking._model_registry.fluent: Run with id 9edcfb2bcfa54450a821275b8865c8e0 has no artifacts at artifact path 'model', registering model based on models:/m-bd8833484ef145c085994345ce5f921f instead
Created version '2' of model 'Advertising_Sales_Model'.


<ModelVersion: aliases=[], creation_timestamp=1788662468373, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788662468373, metrics=None, model_id=None, name='Advertising_Sales_Model', params=None, run_id='9edcfb2bcfa54450a821275b8865c8e0', run_link=None, source='models:/m-bd8833484ef145c085994345ce5f921f', status='READY', status_message=None, tags={}, user_id=None, version=2, workspace='default'>

## Model Aliases

Model aliases gives a nickname to a specific version of the registered model.

Instead of typing exact numbers like Version 1, you just use the nickname.

- **Advertising_Sales_Model**
  - Version 1
  - Version 2
  - Version 3     ← champion 
  - Version 4     ← challenger

**champion means:**
the currently preferred model

**challenger means:**
a new candidate being evaluated as a possible replacement 

In [14]:
from mlflow import MlflowClient

client = MlflowClient()

# 1. Assign an alias to a specific version
# (Sets the alias champion to version 2 of fraud_detector)
client.set_registered_model_alias(
    name = "Advertising_Sales_Model",
    alias="champion",
    version="1"
)

**Loading a registered model**

In [16]:
model= mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion"
)

#Generate new predictions
new_data = pd.DataFrame({
    "TV": [150.0],
    "radio": [25.0],
    "newspaper": [30.0]
})
print(model.predict(new_data))

[15.13020624]
